# Recomendación de artistas con vecinos y ALS

¿Podemos anticipar diez artistas que un usuario todavía no escuchó? Comparamos dos modelos que aprenden del historial de escuchas. Como referencia usamos los artistas con más oyentes distintos. Todas las listas excluyen artistas que el usuario ya escuchó desde el inicio del historial observado, en enero de 2010.

Ejecutar desde la carpeta `mineria/` con `config.py`, el Parquet de eventos limpios y las librerías NumPy, pandas, SciPy, scikit-learn, DuckDB e implicit.

In [21]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from sklearn.preprocessing import StandardScaler, normalize
from implicit.als import AlternatingLeastSquares
from config import conectar, ruta_sql

In [22]:
con = conectar()
con.execute(f"CREATE OR REPLACE VIEW eventos AS SELECT * FROM {ruta_sql('events_clean')}")
VECINOS = [10, 25, 50, 75, 100]
N_ARTISTAS = 5000
TOP = 10

## 1. Preparar el historial y los descubrimientos

En cada corte elegimos los 5.000 artistas con más oyentes del historial disponible. Así conservamos un catálogo amplio y podemos puntuar todos sus candidatos. Para evaluar tomamos hasta 1.000 usuarios con al menos diez artistas conocidos y un descubrimiento del catálogo durante el trimestre siguiente. Esto permite medir qué proporción de sus descubrimientos recuperamos. Los demás usuarios del historial también aportan información al entrenamiento.

In [23]:
def elegir_catalogo(inicio, corte):
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE catalogo AS
        SELECT artist_id, COUNT(DISTINCT user_id) AS oyentes
        FROM eventos WHERE timestamp >= {inicio} AND timestamp < {corte}
        GROUP BY artist_id ORDER BY oyentes DESC, artist_id LIMIT {N_ARTISTAS}
    """)

In [24]:
def contar_escuchas(inicio, corte):
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE pares_pasado AS
        SELECT user_id, artist_id, COUNT(*) AS escuchas
        FROM eventos JOIN catalogo USING (artist_id)
        WHERE timestamp >= {inicio} AND timestamp < {corte}
        GROUP BY user_id, artist_id
    """)
    return con.sql("SELECT * FROM pares_pasado").df()

In [25]:
def descubrimientos(corte, fin):
    return con.sql(f"""
        SELECT DISTINCT user_id, artist_id
        FROM eventos JOIN catalogo USING (artist_id)
        WHERE timestamp >= {corte} AND timestamp < {fin}
        EXCEPT SELECT user_id, artist_id FROM pares_pasado
    """).df()

#### Convertir los conteos a pesos
Cada identificador recibe una posición. Guardamos únicamente los pares observados y sus pesos.

In [26]:
def armar_matriz(pasado):
    filas, usuarios = pd.factorize(pasado.user_id, sort=True)
    columnas, artistas = pd.factorize(pasado.artist_id, sort=True)
    pesos = np.log1p(pasado.escuchas.to_numpy(dtype="float32"))
    matriz = csr_matrix((pesos, (filas, columnas)), shape=(len(usuarios), len(artistas)))
    return matriz, usuarios, artistas

In [27]:
def seleccionar_usuarios(futuro, pasado):
    esperados = futuro.groupby("user_id")["artist_id"].agg(set)
    historial = pasado.groupby("user_id").size()
    elegibles = esperados.index.intersection(historial[historial >= 10].index)
    muestra = esperados.loc[elegibles].sample(n=min(1000, len(elegibles)), random_state=42)
    if muestra.empty:
        raise ValueError("El período no tiene usuarios evaluables.")
    return muestra

In [28]:
def mostrar_cobertura(muestra, inicio, corte, fin):
    con.register("usuarios_muestra", muestra.index.to_frame(index=False, name="user_id"))
    total_nuevos = con.sql(f"""
        SELECT COUNT(*) FROM (
            SELECT user_id, artist_id FROM eventos JOIN usuarios_muestra USING (user_id)
            WHERE timestamp >= {corte} AND timestamp < {fin}
            EXCEPT
            SELECT user_id, artist_id FROM eventos JOIN usuarios_muestra USING (user_id)
            WHERE timestamp >= {inicio} AND timestamp < {corte}
        ) AS descubrimientos
    """).fetchone()[0]
    con.unregister("usuarios_muestra")
    dentro = muestra.map(len)
    print(f"Descubrimientos dentro del catálogo (muestra evaluada): {dentro.sum():,} de {total_nuevos:,} ({dentro.sum()/total_nuevos:.1%})")
    print("Mediana de artistas nuevos del catálogo por usuario:", dentro.median())

In [29]:
def preparar_periodo(corte, fin):
    inicio, corte, fin = [int(pd.Timestamp(f, tz="UTC").timestamp()) for f in ["2010-01-01", corte, fin]]
    elegir_catalogo(inicio, corte)
    pasado = contar_escuchas(inicio, corte)
    matriz, usuarios, artistas = armar_matriz(pasado)
    muestra = seleccionar_usuarios(descubrimientos(corte, fin), pasado)
    mostrar_cobertura(muestra, inicio, corte, fin)
    objetivos = {usuarios.get_loc(u): set(artistas.get_indexer(list(ids))) for u, ids in muestra.items()}
    oyentes = con.sql("SELECT * FROM catalogo").df().set_index("artist_id")["oyentes"]
    return matriz, oyentes.reindex(artistas).to_numpy(), objetivos, artistas, usuarios

## 2. Ver la matriz usuario × artista

Cada casillero guarda `log1p(escuchas)`, es decir, el logaritmo natural de uno más la cantidad de reproducciones. El cero indica que no observamos escuchas. Una escucha pesa 0,693 y diez pesan 2,398. Así conservamos la diferencia entre escuchar una vez y volver muchas veces sin darle diez veces más peso. La matriz dispersa guarda únicamente los pares observados. Las dos tablas siguientes muestran los mismos usuarios y artistas antes y después de la transformación.

In [30]:
val = preparar_periodo("2013-10-01", "2014-01-01")
matriz, _, objetivos, artistas, usuarios = val
filas_ejemplo = list(objetivos)[:5]
oyentes_ejemplo = usuarios.take(filas_ejemplo)
print("Filas (usuarios), columnas (artistas):", matriz.shape)
print("Pares con escuchas:", matriz.nnz)
print(f"Densidad: {matriz.nnz / (matriz.shape[0] * matriz.shape[1]):.2%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Descubrimientos dentro del catálogo (muestra evaluada): 18,256 de 36,588 (49.9%)
Mediana de artistas nuevos del catálogo por usuario: 8.0
Filas (usuarios), columnas (artistas): (54081, 5000)
Pares con escuchas: 8268606
Densidad: 3.06%


In [31]:
# Recuperamos los conteos reales de esos usuarios desde el entrenamiento.
con.register("ejemplo", pd.DataFrame({"user_id": oyentes_ejemplo}))
pares = con.sql("SELECT * FROM pares_pasado JOIN ejemplo USING (user_id)").df()
con.unregister("ejemplo")
conteos = pares.pivot(index="user_id", columns="artist_id", values="escuchas").fillna(0)
seleccion = conteos.sum().nlargest(8).index
print("Cantidad de escuchas antes de la transformación")
display(conteos.loc[oyentes_ejemplo, seleccion].astype(int))

Cantidad de escuchas antes de la transformación


artist_id,93,13037,1915,9873,4115,6031,4276,13285
5954302,0,0,0,0,0,408,0,0
18732172,0,0,0,0,0,0,0,0
7631411,0,0,0,0,1,0,0,0
4065372,754,0,0,0,419,0,0,0
1048362,0,626,493,446,2,0,378,350


In [32]:
# Solo convertimos a tabla visible este pequeño recorte de la matriz CSR.
columnas_ejemplo = artistas.get_indexer(seleccion)
pesos = matriz[filas_ejemplo][:, columnas_ejemplo].toarray()
print("Pesos que usa el modelo: log(1 + escuchas)")
display(pd.DataFrame(pesos, index=oyentes_ejemplo, columns=seleccion).round(3))
assert np.allclose(pesos, np.log1p(conteos.loc[oyentes_ejemplo, seleccion]))

Pesos que usa el modelo: log(1 + escuchas)


artist_id,93,13037,1915,9873,4115,6031,4276,13285
5954302,0.000,0.000,0.000,0.000,0.000,6.014,0.000,0.000
18732172,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7631411,0.000,0.000,0.000,0.000,0.693,0.000,0.000,0.000
4065372,6.627,0.000,0.000,0.000,6.040,0.000,0.000,0.000
1048362,0.000,6.441,6.203,6.103,1.099,0.000,5.938,5.861


## 3. ¿Cómo se puntúa a cada artista?

**Popularidad** ordena por cantidad de oyentes distintos en el entrenamiento. **Vecinos** compara las filas de la matriz logarítmica por similitud coseno. Cada usuario parecido aporta sus artistas y los más similares tienen más peso.

In [33]:
def puntuar_vecinos(parametro, matriz, filas):
    buscador = NearestNeighbors(metric="cosine", algorithm="brute").fit(matriz)
    distancias, indices = buscador.kneighbors(matriz[filas], n_neighbors=parametro + 1)
    puntajes = []
    for u, dist, ids in zip(filas, distancias, indices):
        otros = ids != u
        vecinos = ids[otros][:parametro]
        similitud = np.maximum(0, 1 - dist[otros][:parametro])
        puntajes.append(matriz[vecinos].T @ similitud)
    return np.asarray(puntajes)

**ALS para datos implícitos** aprende un vector por usuario y otro por artista. Su producto da el puntaje de recomendación. Cada par observado indica interés y recibe una confianza de `1 + log1p(escuchas)`. El código suma uno solo a los valores almacenados en una copia dispersa. Los pares sin escuchas siguen ausentes de la matriz de entrada. Probamos cuántos factores necesita cada vector y mantenemos fijas las otras opciones para interpretar el efecto de ese único cambio.

In [34]:
def puntuar_als(parametro, matriz, filas):
    confianza = matriz.copy()
    confianza.data += 1
    als = AlternatingLeastSquares(factors=parametro, regularization=0.1,
                                  iterations=30, alpha=1, random_state=42,
                                  use_gpu=False, num_threads=4)
    als.fit(confianza, show_progress=False)
    return als.user_factors[filas] @ als.item_factors.T

In [35]:
def calcular_puntajes(modelo, parametro, matriz, popularidad, filas):
    if modelo == "Popularidad":
        return np.broadcast_to(popularidad, (len(filas), len(popularidad)))
    if modelo == "Vecinos":
        return puntuar_vecinos(parametro, matriz, filas)
    if modelo == "ALS":
        return puntuar_als(parametro, matriz, filas)
    raise ValueError(modelo)

## 4. Evaluar las cinco recomendaciones

Para cada usuario, **Precision@5** divide los aciertos por cinco, **Recall@5** los divide por sus descubrimientos dentro del catálogo y **Hit Rate@5** registra si hubo al menos uno. **NDCG@5** premia que los aciertos aparezcan primero y compara ese orden con el mejor posible para ese usuario. Promediamos por usuario. La cobertura del catálogo indica qué porcentaje de los 5.000 artistas apareció en alguna lista.

In [36]:
def metricas_usuario(nuevos, esperados):
    descuentos = 1 / np.log2(np.arange(2, TOP + 2))
    relevantes = np.array([a in esperados for a in nuevos])
    aciertos = relevantes.sum()
    dcg = np.dot(relevantes, descuentos[:len(nuevos)])
    ideal = descuentos[:min(TOP, len(esperados))].sum()
    return [aciertos / TOP, aciertos / len(esperados), aciertos > 0, dcg / ideal]

In [37]:
def evaluar(modelo, parametro, datos):
    matriz, popularidad, objetivos, artistas, _ = datos
    filas = list(objetivos)
    puntajes = calcular_puntajes(modelo, parametro, matriz, popularidad, filas)
    recomendados = set()
    medidas = []
    for u, puntos in zip(filas, puntajes):
        orden = np.lexsort((artistas, -popularidad, -puntos))
        nuevos = orden[~np.isin(orden, matriz[u].indices)][:TOP]
        recomendados.update(nuevos)
        medidas.append(metricas_usuario(nuevos, objetivos[u]))
    precision, recall, hit, ndcg = np.mean(medidas, axis=0)
    return {"modelo": modelo, "parametro": parametro, "usuarios": len(filas),
            "catalogo_recomendado_pct": 100 * len(recomendados) / len(artistas), "precision@5": precision,
            "recall@5": recall, "hit_rate@5": hit, "ndcg@5": ndcg}

## 5. Elegir los parámetros en validación

Con el historial anterior a octubre de 2013 recomendamos para octubre–diciembre. Para vecinos probamos cinco tamaños de vecindario y para ALS cinco cantidades de factores. En ALS fijamos la regularización en 0,1, las iteraciones en 30, alpha en 1 y la semilla en 42, como en la corrida original. Elegimos por NDCG@5 porque nos interesa acertar y ubicar esos artistas cerca del inicio.

In [38]:
configuraciones = {"Vecinos": VECINOS, "ALS": [8, 16, 24, 32, 48]}

In [39]:
resultados = [evaluar("Popularidad", None, val)]
elegidos = {}
for modelo, valores in configuraciones.items():
    pruebas = [evaluar(modelo, v, val) for v in valores]
    resultados.extend(pruebas)
    elegidos[modelo] = max(pruebas, key=lambda r: r["ndcg@5"])["parametro"]
display(pd.DataFrame(resultados).round(4))
print("Parámetros elegidos:", elegidos)

,modelo,parametro,usuarios,catalogo_recomendado_pct,precision@5,recall@5,hit_rate@5,ndcg@5
0,Popularidad,NaN,1000,1.22,0.0350,0.0264,0.232,0.0432
1,Vecinos,10.0,1000,45.36,0.0726,0.0625,0.400,0.0914
2,Vecinos,25.0,1000,40.46,0.0762,0.0659,0.418,0.0982
3,Vecinos,50.0,1000,36.42,0.0779,0.0675,0.424,0.1007
4,Vecinos,75.0,1000,33.72,0.0773,0.0693,0.432,0.1011
5,Vecinos,100.0,1000,32.20,0.0785,0.0707,0.430,0.1027
6,ALS,8.0,1000,11.46,0.0631,0.0574,0.363,0.0831
7,ALS,16.0,1000,18.46,0.0726,0.0660,0.399,0.0967
8,ALS,24.0,1000,22.14,0.0734,0.0664,0.406,0.0983
9,ALS,32.0,1000,25.20,0.0750,0.0663,0.406,0.0988


Parámetros elegidos: {'Vecinos': 100, 'ALS': 48}


**Resultados de la corrida original.** NDCG@5 en validación para cada configuración.

| Vecinos | NDCG@5 | Factores ALS | NDCG@5 |
|---:|---:|---:|---:|
| 10 | 0,0902 | 8 | 0,0822 |
| 25 | 0,0978 | 16 | 0,0985 |
| 50 | 0,0993 | 24 | 0,1009 |
| 75 | 0,1008 | 32 | 0,1003 |
| **100** | **0,1018** | **48** | **0,1030** |

## 6. Probar en el trimestre siguiente

Actualizamos el entrenamiento con las escuchas hasta diciembre y aplicamos los parámetros elegidos. Contrastamos las listas con los nuevos artistas escuchados entre enero y marzo de 2014.

In [40]:
test = preparar_periodo("2014-01-01", "2014-04-01")
final = [evaluar("Popularidad", None, test)]
for modelo, valor in elegidos.items():
    final.append(evaluar(modelo, valor, test))
print("Matriz de entrenamiento (usuarios, artistas):", test[0].shape)
print("Pares con escuchas:", test[0].nnz, "| Usuarios evaluados:", len(test[2]))
display(pd.DataFrame(final).sort_values("ndcg@5", ascending=False).round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Descubrimientos dentro del catálogo (muestra evaluada): 16,376 de 32,955 (49.7%)
Mediana de artistas nuevos del catálogo por usuario: 7.5
Matriz de entrenamiento (usuarios, artistas): (54663, 5000)
Pares con escuchas: 8819918 | Usuarios evaluados: 1000


,modelo,parametro,usuarios,catalogo_recomendado_pct,precision@5,recall@5,hit_rate@5,ndcg@5
2,ALS,48.0,1000,30.34,0.0700,0.0652,0.394,0.0896
1,Vecinos,100.0,1000,31.38,0.0666,0.0597,0.380,0.0873
0,Popularidad,NaN,1000,1.32,0.0290,0.0220,0.198,0.0354


### Resultados de la corrida original sobre 1.000 usuarios

| Método | Precision@5 | Recall@5 | Hit Rate@5 | NDCG@5 |
|---|---:|---:|---:|---:|
| Popularidad | 3,40% | 1,22% | 13,1% | 0,0363 |
| Vecinos (100) | 7,82% | 3,55% | 27,5% | 0,0891 |
| ALS (48 factores) | 7,66% | 3,85% | 27,4% | 0,0871 |

Ambos modelos mejoraron sobre popularidad. Vecinos acertó al menos un artista para 275 personas y ALS para 274, frente a 131 con popularidad. Vecinos obtuvo el mayor NDCG@5 en este trimestre. ALS recuperó en promedio una proporción algo mayor de los descubrimientos de cada usuario. Estos resultados describen a los usuarios evaluados y las novedades que estaban dentro del catálogo.